1. Modular Component Library
This section handles authentication and the initial profiling of the downloaded Kaggle datasets.

In [0]:
import os
import shutil
import kagglehub
import pandas as pd
import gc
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

# --- 1. CONFIGURATION ---
KAGGLE_SCOPE = "KaggleCreds"
DATASET_HANDLE = "zillow/zecon"
VOLUME_BASE_PATH = "/Volumes/data_landing/data_raw"
SIZE_LIMIT_BYTES = 500 * 1024 * 1024 

def initialize_kaggle_auth(scope):
    os.environ['KAGGLE_USERNAME'] = dbutils.secrets.get(scope=scope, key="username")
    os.environ['KAGGLE_KEY'] = dbutils.secrets.get(scope=scope, key="key")

def generate_xml_spark_chunk(volume_source_path, dest_dir, start_offset, dataset_name):
    """
    Handles high-overhead XML using Spark with encoding fix.
    """
    print(f"  - Starting Spark XML generation (Chunk 4)...")
    
    # FIX: Added encoding option for Spark
    df = spark.read.option("header", "true") \
              .option("encoding", "ISO-8859-1") \
              .csv(volume_source_path)
              
    for col_name in df.columns:
        df = df.withColumn(col_name, F.col(col_name).cast(StringType()))

    window_spec = Window.orderBy(F.monotonically_increasing_id())
    chunk4_df = df.withColumn("row_num", F.row_number().over(window_spec)) \
        .filter(F.col("row_num") > start_offset) \
        .drop("row_num")
    
    chunk4_df = chunk4_df \
        .withColumn("load_dt", F.current_timestamp().cast(StringType())) \
        .withColumn("source", F.lit(f"{dataset_name}.csv"))

    temp_xml_path = f"{dest_dir}/temp_xml_{dataset_name}"
    (chunk4_df.coalesce(1).write
     .format("xml")
     .option("rootTag", "ZillowData")
     .option("rowTag", "Record")
     .mode("overwrite")
     .save(temp_xml_path))

    try:
        files = [f for f in os.listdir(temp_xml_path) if f.startswith("part-") and f.endswith(".xml")]
        if files:
            shutil.move(os.path.join(temp_xml_path, files[0]), f"{dest_dir}/chunk4.xml")
            shutil.rmtree(temp_xml_path)
            print(f"  - Successfully dumped chunk4.xml")
    except Exception as e:
        print(f"  - XML move failed: {str(e)}")

RAM-Optimized Chunking Logic

This section contains the core logic for splitting large files. Note the updated save_chunk logic for XML, which now constructs a composite file containing both <schema> and <data> nodes.

In [0]:
def process_large_file_split(volume_source_path, dest_dir, total_rows, dataset_name):
    """
    Handles 4-way split with encoding fix for Pandas.
    """
    os.makedirs(dest_dir, exist_ok=True)
    c1_size = int(total_rows * 0.40)
    c2_size = int(total_rows * 0.40)
    c3_size = int(total_rows * 0.10)

    for i, (skip, nrows, fmt, name) in enumerate([
        (0, c1_size, 'csv', 'chunk1.csv'),
        (c1_size, c2_size, 'csv', 'chunk2.csv'),
        (c1_size + c2_size, c3_size, 'json', 'chunk3.json')
    ]):
        skip_range = range(1, skip + 1) if skip > 0 else None
        target = f"{dest_dir}/{name}"
        
        # FIX: Added encoding='ISO-8859-1' for Pandas
        df = pd.read_csv(volume_source_path, skiprows=skip_range, nrows=nrows, 
                         low_memory=False, dtype=str, encoding='ISO-8859-1')
        
        df['load_dt'] = pd.Timestamp.now()
        df['source'] = f"{dataset_name}.csv"
        
        if fmt == 'csv': df.to_csv(target, index=False)
        else: df.to_json(target, orient='records')
        print(f"  - Generated {name}")
        del df
        gc.collect()
    
    generate_xml_spark_chunk(volume_source_path, dest_dir, (c1_size + c2_size + c3_size), dataset_name)

def process_small_file_direct(volume_source_path, dest_volume_dir, dataset_name):
    """
    Small file processing with encoding fix.
    """
    print(f"  - Finalizing small file: {dataset_name}")
    # FIX: Added encoding='ISO-8859-1'
    df = pd.read_csv(volume_source_path, low_memory=False, dtype=str, encoding='ISO-8859-1')
    df['load_dt'] = pd.Timestamp.now()
    df['source'] = f"{dataset_name}.csv"
    
    final_path = f"{dest_volume_dir}/{dataset_name}.csv"
    df.to_csv(final_path, index=False)
    
    if volume_source_path != final_path:
        os.remove(volume_source_path)
    
    del df
    gc.collect()

Main Execution Orchestrator
This section coordinates the authentication, download, profiling, and final execution of the chunking job.

In [0]:
def run_data_chunking_job():
    initialize_kaggle_auth(KAGGLE_SCOPE)
    
    print(f"Downloading dataset: {DATASET_HANDLE}")
    local_cache_path = kagglehub.dataset_download(DATASET_HANDLE)
    all_files = [f for f in os.listdir(local_cache_path) if f.lower().endswith('.csv')]
    
    for fname in all_files:
        local_path = os.path.join(local_cache_path, fname)
        clean_name = fname.lower().replace('.csv', '')
        dest_volume_dir = f"{VOLUME_BASE_PATH}/{clean_name}"
        os.makedirs(dest_volume_dir, exist_ok=True)
        
        intermediate_vol_path = f"{dest_volume_dir}/raw_temp_{fname.lower()}"
        shutil.move(local_path, intermediate_vol_path)
        
        fsize = os.path.getsize(intermediate_vol_path)

        if fsize > SIZE_LIMIT_BYTES:
            print(f"\n[LARGE FILE] Splitting {fname} (Size: {fsize/(1024**2):.2f} MB)...")
            # FIX: Added encoding='ISO-8859-1' and 'ignore' errors for line counting
            with open(intermediate_vol_path, 'r', encoding='ISO-8859-1', errors='ignore') as f:
                total_rows = sum(1 for _ in f) - 1
            
            process_large_file_split(intermediate_vol_path, f"{dest_volume_dir}/chunks", total_rows, clean_name)
            os.remove(intermediate_vol_path) 
        else:
            print(f"\n[SMALL FILE] Processing {fname}...")
            process_small_file_direct(intermediate_vol_path, dest_volume_dir, clean_name)

    print("\n[SUCCESS] Pipeline Finished.")

if __name__ == "__main__":
    run_data_chunking_job()